In [1]:
import os
import time
import numpy as np
import polars as pl
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
# from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ART

from art.attacks.evasion import ProjectedGradientDescent, FastGradientMethod
from art.estimators.classification import TensorFlowV2Classifier, SklearnClassifier, XGBoostClassifier, LightGBMClassifier, CatBoostARTClassifier, KerasClassifier

I0000 00:00:1778004432.840736 2927326 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778004432.889160 2927326 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778004434.077923 2927326 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packa

In [2]:
# Carga Dataset UNSW-NB15

path_train = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_train_NUSW_redux.csv"
path_test  = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_test_NUSW_redux.csv"
TARGET_COL = "attack_cat"

df_train = pl.read_csv(path_train)
df_test  = pl.read_csv(path_test)

y_train = (
    df_train.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

y_test = (
    df_test.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

x_train = df_train.drop(TARGET_COL)
x_test  = df_test.drop(TARGET_COL)

X_full_train = x_train.to_numpy()
X_test_np = x_test.to_numpy()
y_full_train = y_train.to_numpy()
y_test_np = y_test.to_numpy()

y_full_train_01 = ((y_full_train + 1) // 2).astype(np.int8)
y_test_np01 = ((y_test_np + 1) // 2).astype(np.int8)

print(f"Train: {X_full_train.shape} | Test: {X_test_np.shape}")
print("Tras convertir -1/1 a 0/1, la clase 0 corresponde a Ataque y la clase 1 a Normal.")

HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0
TRAIN_DEVICE = "/GPU:0" if HAS_GPU else "/CPU:0"
INFER_DEVICE = "/CPU:0"

Train: (175341, 12) | Test: (82332, 12)
Tras convertir -1/1 a 0/1, la clase 0 corresponde a Ataque y la clase 1 a Normal.


In [3]:
# Modelos ganadores de UNSW-NB15

RF_CONFIG = {"n_estimators": 50, "max_depth": 23}
XGB_CONFIG = {"n_estimators": 200, "max_depth": 11, "learning_rate": 0.1}
LGBM_CONFIG = {"n_estimators": 100, "num_leaves": 145, "max_depth": 12, "learning_rate": 0.1}
CATBOOST_CONFIG = {"iterations": 500, "depth": 10, "learning_rate": 0.1}

SVM_C = 0.000187
MLP_INPUT_DIM = X_full_train.shape[1]
CNN1D_CONFIG = {"nf": 64, "k": 5, "d": 48}

def build_mlp_model(input_dim):
    model = keras.Sequential([
        keras.layers.InputLayer(input_shape=(input_dim,)),
        keras.layers.Dense(96, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

DEFAULT_CNN_DROPOUT = 0.2

def build_cnn1d_model(n_features, n_filters, kernel_size, dense_units, dropout_rate=DEFAULT_CNN_DROPOUT):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features, 1)),
        keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.MaxPooling1D(pool_size=2),
        keras.layers.Conv1D(filters=max(16, n_filters // 2), kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.GlobalMaxPooling1D(),
        keras.layers.Dense(dense_units, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def clone_keras_model_to_cpu(builder_fn, trained_model, *builder_args):
    with tf.device(INFER_DEVICE):
        cpu_model = builder_fn(*builder_args)
        cpu_model.set_weights(trained_model.get_weights())
    return cpu_model

In [4]:
# Entrenamiento modelos con su dataset

X_test_np_arr = np.array(X_test_np)

print("Entrenando Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=RF_CONFIG["n_estimators"],
    max_depth=RF_CONFIG["max_depth"],
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_full_train, y_full_train_01)

# ==========================================

print("\nEntrenando XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=XGB_CONFIG["n_estimators"],
    max_depth=XGB_CONFIG["max_depth"],
    learning_rate=XGB_CONFIG["learning_rate"],
    tree_method="hist",
    device="cuda" if HAS_GPU else "cpu",
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_full_train, y_full_train_01)

# ==========================================

print("\nEntrenando LightGBM...")
lgbm_model = LGBMClassifier(
    n_estimators=LGBM_CONFIG["n_estimators"],
    num_leaves=LGBM_CONFIG["num_leaves"],
    max_depth=LGBM_CONFIG["max_depth"],
    learning_rate=LGBM_CONFIG["learning_rate"],
    device_type="gpu" if HAS_GPU else "cpu",
    n_jobs=-1,
    random_state=42,
    verbosity=-1
)
lgbm_model.fit(X_full_train, y_full_train_01)

# ==========================================
print("\nEntrenando CatBoost...")
cat_model = CatBoostClassifier(
    iterations=CATBOOST_CONFIG["iterations"],
    depth=CATBOOST_CONFIG["depth"],
    learning_rate=CATBOOST_CONFIG["learning_rate"],
    random_seed=42,
    logging_level="Silent",
    task_type="GPU" if HAS_GPU else "CPU"
)
cat_model.fit(X_full_train, y_full_train_01)

Entrenando Random Forest...

Entrenando XGBoost...

Entrenando LightGBM...

Entrenando CatBoost...


CatBoostClassifier(depth=10, iterations=500, learning_rate=0.1, logging_level='Silent', random_seed=42, task_type='GPU')

In [5]:
# ==========================================

print("\nEntrenando Linear SVM...")
svm_model = make_pipeline(
    StandardScaler(),
    LinearSVC(C=SVM_C, dual=False, random_state=42, max_iter=2000)
)
svm_model.fit(X_full_train, y_full_train_01)

# ==========================================

print("\nEntrenando MLP...")
tf.keras.backend.clear_session()
mlp_scaler = StandardScaler()
X_train_scaled_mlp = mlp_scaler.fit_transform(X_full_train)
X_test_scaled_mlp = mlp_scaler.transform(X_test_np_arr)
with tf.device(INFER_DEVICE):
    mlp_model = build_mlp_model(MLP_INPUT_DIM)
    mlp_early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    mlp_model.fit(
        X_train_scaled_mlp,
        y_full_train_01,
        validation_split=0.1,
        epochs=40,
        batch_size=2048,
        callbacks=[mlp_early],
        verbose=0
    )

# ==========================================

print("\nEntrenando CNN-1D...")
tf.keras.backend.clear_session()
cnn_scaler = MinMaxScaler()
X_train_scaled_cnn = cnn_scaler.fit_transform(X_full_train)
X_test_scaled_cnn = cnn_scaler.transform(X_test_np_arr)
X_train_tabular_cnn = X_train_scaled_cnn.reshape(X_train_scaled_cnn.shape[0], X_train_scaled_cnn.shape[1], 1)
X_test_tabular_cnn = X_test_scaled_cnn.reshape(X_test_scaled_cnn.shape[0], X_test_scaled_cnn.shape[1], 1)
with tf.device(INFER_DEVICE):
    cnn_model = build_cnn1d_model(X_train_tabular_cnn.shape[1], CNN1D_CONFIG["nf"], CNN1D_CONFIG["k"], CNN1D_CONFIG["d"])
    cnn_early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    cnn_model.fit(
        X_train_tabular_cnn,
        y_full_train_01,
        validation_split=0.1,
        epochs=20,
        batch_size=1024,
        callbacks=[cnn_early],
        verbose=0
    )


Entrenando Linear SVM...

Entrenando MLP...


I0000 00:00:1778004501.501972 2927326 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 42726 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:4a:00.0, compute capability: 8.9
I0000 00:00:1778004501.505485 2927326 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 43485 MB memory:  -> device: 1, name: NVIDIA L40S, pci bus id: 0000:61:00.0, compute capability: 8.9
I0000 00:00:1778004501.506936 2927326 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 43485 MB memory:  -> device: 2, name: NVIDIA L40S, pci bus id: 0000:ca:00.0, compute capability: 8.9
I0000 00:00:1778004501.509268 2927326 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 43485 MB memory:  -> device: 3, name: NVIDIA L40S, pci bus id: 0000:e1:00.0, compute capability: 8.9
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/keras/src/layers/core/input_layer.py:


Entrenando CNN-1D...


Una vez que los modelos ya están entrenados, ya podemos entrar en la fase de Evaluación Adversaria. Primero vamos a ver que nuestro modelo recién entrenado rinda bien con x_test limpio

In [6]:
# Evaluamos en test MLP

mlp_model_cpu = clone_keras_model_to_cpu(build_mlp_model, mlp_model, MLP_INPUT_DIM)
def mlp_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)
def mlp_predict_scores(X):
    with tf.device(INFER_DEVICE):
        return mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()

# F1-score de referencia en test para MLP
y_test_pred_mlp = mlp_predict_labels(X_test_scaled_mlp)
mlp_f1 = f1_score(y_test_np01, y_test_pred_mlp)
print(f"\nMLP F1-score en test: {mlp_f1:.4f}")

/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



MLP F1-score en test: 0.6794


In [7]:
# Extraemos restricciones tabulares a partir del train original
feature_names = x_train.columns

tabular_constraints_df = pl.DataFrame({
    "feature": feature_names,
    "min": x_train.min().row(0),
    "max": x_train.max().row(0),
})

feature_mins = tabular_constraints_df["min"].to_numpy().astype(np.float32)
feature_maxs = tabular_constraints_df["max"].to_numpy().astype(np.float32)

tabular_constraints = {
    feature: {"min": float(min_val), "max": float(max_val)}
    for feature, min_val, max_val in zip(feature_names, feature_mins, feature_maxs)
}

# 1. Restricciones para modelos que trabajan en espacio original
clip_values_raw = (feature_mins, feature_maxs)

# 2. Restricciones para el MLP, que trabaja con StandardScaler
feature_mins_mlp = mlp_scaler.transform(feature_mins.reshape(1, -1)).ravel().astype(np.float32)
feature_maxs_mlp = mlp_scaler.transform(feature_maxs.reshape(1, -1)).ravel().astype(np.float32)
clip_values_mlp = (feature_mins_mlp, feature_maxs_mlp)

# 3. Restricciones para la CNN, que trabaja con MinMaxScaler
feature_mins_cnn = cnn_scaler.transform(feature_mins.reshape(1, -1)).ravel().astype(np.float32)
feature_maxs_cnn = cnn_scaler.transform(feature_maxs.reshape(1, -1)).ravel().astype(np.float32)
clip_values_cnn = (feature_mins_cnn, feature_maxs_cnn)

print("Restricciones tabulares extraidas para UNSW-NB15:")
display(tabular_constraints_df)

print(clip_values_mlp)

Restricciones tabulares extraidas para UNSW-NB15:


feature,min,max
str,f64,f64
"""dur""",0.0,59.999989
"""sinpkt""",0.0,84371.496
"""dinpkt""",0.0,56716.824
"""spkts""",1.0,9616.0
"""dpkts""",0.0,10974.0
…,…,…
"""rate""",0.0,1.0000e6
"""smean""",28.0,1504.0
"""dmean""",0.0,1458.0


(array([-0.20977475, -0.1361428 , -0.08937003, -0.14098223, -0.17204736,
       -0.05044967, -0.10392289, -0.57681924, -0.5313342 , -0.48070285,
       -0.48434597, -0.50301373], dtype=float32), array([  9.049153 ,  11.513798 ,  57.369225 ,  70.09933  ,  99.358185 ,
        74.135994 , 101.91599  ,   5.469112 ,   6.6800365,   5.1635404,
        47.911243 ,  37.04389  ], dtype=float32))


In [8]:
# Hay que tener en cuenta que si tuviéramos columnas One Hot Encoding, habría que asegurarse de que las restricciones 
# se apliquen correctamente y que ART no me genere un TCP = 0.62, ya que no tiene sentido que un valor One Hot Encoding tenga un valor intermedio entre 0 y 1. 
# En ese caso habría que indicarle a ART que esas columnas solo pueden tomar valores 0 o 1, y no valores continuos.

# ==========================================
# FASE 1. ENVOLVER EL MODELO (Caja Blanca)
# ==========================================

print("Envolviendo el modelo MLP en ART con restricciones tabulares...")

clasificador_art_mlp = KerasClassifier(
    model=mlp_model_cpu, 
    clip_values=clip_values_mlp, 
    use_logits=False
)

Envolviendo el modelo MLP en ART con restricciones tabulares...


In [24]:
# ===================================================
# FASE 2. FUERZA DE ATAQUE POR CARACTERÍSTICA (eps)
# ===================================================

print("Configurando la fuerza del ataque (eps) por característica...")

# Definimos la fuerza base del ataque como un 20% del rango permitido
# por cada característica en el espacio escalado del MLP.
eps_base = 0.01 # Cuánto permitimos que se modifique cada característica

feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]
eps_vector = (eps_base * feature_ranges_mlp).astype(np.float32)
print(eps_vector)

# Definimos un paso por iteración como una fracción del presupuesto total.
eps_step_vector = np.where(eps_vector > 0, eps_vector / 4.0, 0.0).astype(np.float32) # El tamaño del paso
print(eps_step_vector)


Configurando la fuerza del ataque (eps) por característica...
[0.09258928 0.1164994  0.5745859  0.70240307 0.9953023  0.74186444
 1.0201991  0.06045931 0.07211371 0.05644243 0.48395586 0.37546906]
[0.02314732 0.02912485 0.14364648 0.17560077 0.24882558 0.18546611
 0.25504977 0.01511483 0.01802843 0.01411061 0.12098897 0.09386726]


In [27]:
# =======================================================
# FASE 3. LANZAR ATAQUE PGD CON RESTRICCIONES TABULARES
# =======================================================

print("Lanzando ataque PGD...")

# Configuramos PGD con el vector eps personalizado
ataque_pgd = ProjectedGradientDescent(
    estimator=clasificador_art_mlp,
    eps=eps_vector,
    eps_step=eps_step_vector,
    max_iter=20,
    batch_size=20
)

# Atacamos el mismo conjunto y en el mismo espacio que usa el MLP
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

# Generamos el dataset adversario
x_test_adversario = ataque_pgd.generate(x=x_test_mlp_attack)
print("¡Tráfico adversario generado con éxito!")

Lanzando ataque PGD...


PGD - Random Initializations:   0%|          | 0/1 [00:00<?, ?it/s]

PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.60it/s]


¡Tráfico adversario generado con éxito!


In [28]:
# COMPARAMOS EL F1-SCORE EN TEST LIMPIO Y EN TEST ADVERSARIO PARA EL MLP

y_test_pred_mlp_clean = mlp_predict_labels(X_test_scaled_mlp)
mlp_f1_clean = f1_score(y_test_np01, y_test_pred_mlp_clean)
print(f"\nMLP F1-score en test limpio: {mlp_f1_clean:.4f}")

y_test_pred_mlp_adv = mlp_predict_labels(x_test_adversario)
mlp_f1_adv = f1_score(y_test_np01, y_test_pred_mlp_adv)
print(f"MLP F1-score en test con tráfico adversario: {mlp_f1_adv:.4f}")



MLP F1-score en test limpio: 0.6794
MLP F1-score en test con tráfico adversario: 0.0635


In [13]:
# ========================================================
# FASE 4. LANZAR ATAQUE FGSM
# ========================================================

print("Lanzando ataque FGSM...")

ataque_fgsm = FastGradientMethod(
    estimator=clasificador_art_mlp,
    eps=eps_vector,
    eps_step=eps_step_vector,
    batch_size=128,
)

# Usamos el mismo conjunto escalado que consume el MLP.
x_test_adversario_fgsm = ataque_fgsm.generate(x=x_test_mlp_attack)
print("¡Tráfico adversario FGSM generado con éxito!")


Lanzando ataque FGSM...
¡Tráfico adversario FGSM generado con éxito!


In [31]:
# COMPARAMOS ACCURACY Y F1-SCORE EN TEST LIMPIO Y EN TEST ADVERSARIO PARA EL MLP

y_test_pred_mlp_clean = mlp_predict_labels(X_test_scaled_mlp)
mlp_acc_clean = accuracy_score(y_test_np01, y_test_pred_mlp_clean)
mlp_f1_clean = f1_score(y_test_np01, y_test_pred_mlp_clean)

print(f"\nMLP Accuracy en test limpio: {mlp_acc_clean:.4f}")
print(f"MLP F1-score en test limpio: {mlp_f1_clean:.4f}")

y_test_pred_mlp_adv = mlp_predict_labels(x_test_adversario)
mlp_acc_adv = accuracy_score(y_test_np01, y_test_pred_mlp_adv)
mlp_f1_adv = f1_score(y_test_np01, y_test_pred_mlp_adv)

print(f"\nMLP Accuracy en test adversario: {mlp_acc_adv:.4f}")
print(f"MLP F1-score en test adversario: {mlp_f1_adv:.4f}")

y_test_pred_mlp_fgsm = mlp_predict_labels(x_test_adversario_fgsm)
mlp_acc_fgsm = accuracy_score(y_test_np01, y_test_pred_mlp_fgsm)
mlp_f1_fgsm = f1_score(y_test_np01, y_test_pred_mlp_fgsm)

print(f"\nMLP Accuracy en test adversario FGSM: {mlp_acc_fgsm:.4f}")
print(f"MLP F1-score en test adversario FGSM: {mlp_f1_fgsm:.4f}")


MLP Accuracy en test limpio: 0.7781
MLP F1-score en test limpio: 0.6794

MLP Accuracy en test adversario: 0.5651
MLP F1-score en test adversario: 0.0635

MLP Accuracy en test adversario FGSM: 0.5613
MLP F1-score en test adversario FGSM: 0.0946


In [32]:
# RF en test limpio
y_test_pred_rf_clean = rf_model.predict(X_test_np)

rf_acc_clean = accuracy_score(y_test_np01, y_test_pred_rf_clean)
rf_f1_clean = f1_score(y_test_np01, y_test_pred_rf_clean)

print(f"RF Accuracy en test limpio: {rf_acc_clean:.4f}")
print(f"RF F1-score en test limpio: {rf_f1_clean:.4f}")

# Transferencia desde el MLP al RF
# Adversarios generados sobre el MLP, convertidos al espacio original
x_test_adversario_raw = mlp_scaler.inverse_transform(x_test_adversario)

# RF sobre tráfico adversario transferido desde el MLP
y_test_pred_rf_adv = rf_model.predict(x_test_adversario_raw)

rf_acc_adv = accuracy_score(y_test_np01, y_test_pred_rf_adv)
rf_f1_adv = f1_score(y_test_np01, y_test_pred_rf_adv)

print(f"RF Accuracy en test adversario transferido: {rf_acc_adv:.4f}")
print(f"RF F1-score en test adversario transferido: {rf_f1_adv:.4f}")


RF Accuracy en test limpio: 0.8665
RF F1-score en test limpio: 0.8320
RF Accuracy en test adversario transferido: 0.5659
RF F1-score en test adversario transferido: 0.0969
